# ImprintQAI Finance — Phase-0 Step-1
**Intraday Imprint Field — Feasibility Test (1-day minute-level)**

**What this notebook does:**
- Pulls 1-day intraday data for AAPL (tries 1m; falls back to 5m/1d if not available).
- Computes minute returns and classifies each minute as up/down (binary label: 1/0).
- Trains the ImprintField model as a binary classifier to predict direction from OHLCV features.
- Plots training loss and reports accuracy.

**Notes:**
- Run the first cell to install required packages. This will work on free Colab CPU runtime.
- If minute-level data isn't available from Yahoo for the day, the notebook falls back to a coarser interval
  or creates a small synthetic intraday split so the experiment still runs.


In [ ]:
# Install minimal dependencies (runs in Colab)
!pip install -q yfinance torch matplotlib pandas tqdm


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)


In [ ]:
# ImprintField implementation (same as in src/core)
class ImprintField(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, imprint_strength=0.15):
        super().__init__()
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, 1)
        self.imprint_strength = imprint_strength

    def forward(self, x):
        encoded = torch.tanh(self.encoder(x))
        imprint = encoded * self.imprint_strength
        output = self.decoder(encoded + imprint)
        return output

class BinaryImprint(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, imprint_strength=0.15):
        super().__init__()
        self.field = ImprintField(input_dim, hidden_dim, imprint_strength)
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        return self.sigmoid(self.field(x))


In [ ]:
# Data loader: try minute data then fallback
def load_intraday(ticker='AAPL'):
    intervals = ['1m', '5m', '15m', '1d']
    data = None
    for interval in intervals:
        try:
            df = yf.download(ticker, period='1d', interval=interval, progress=False)
            if df is None or df.empty:
                continue
            # If interval is 1d and previous intervals failed, create synthetic intraday slices
            df = df.copy()
            df['interval'] = interval
            return df
        except Exception as e:
            # continue trying next interval
            # print("interval", interval, "failed:", e)
            continue
    # As a final fallback, create a tiny synthetic intraday dataset
    times = pd.date_range(end=pd.Timestamp.today().normalize(), periods=60, freq='T')
    df = pd.DataFrame({
        'Open': np.linspace(100, 101, len(times)) + np.random.randn(len(times))*0.05,
        'High': np.linspace(100, 101.1, len(times)) + np.random.randn(len(times))*0.05,
        'Low': np.linspace(99.9, 100.9, len(times)) + np.random.randn(len(times))*0.05,
        'Close': np.linspace(100, 101, len(times)) + np.random.randn(len(times))*0.05,
        'Volume': np.random.randint(1000,5000,size=len(times))
    }, index=times)
    df['interval'] = 'synthetic'
    return df

df = load_intraday('AAPL')
print('Loaded data with interval:', df.get('interval', 'unknown').iloc[0] if 'interval' in df.columns else 'unknown')
df.head()


In [ ]:
# Prepare features and binary labels (next-minute direction)
df = df.dropna().copy()
# Use OHLCV as features; compute returns
df['Return'] = df['Close'].pct_change().fillna(0)
# Label: next step return positive -> 1 else 0
df['NextReturn'] = df['Return'].shift(-1).fillna(0)
df['Direction'] = (df['NextReturn'] > 0).astype(int)

# Drop last row where NextReturn was filled from NaN
df = df.iloc[:-1]

features = df[['Open','High','Low','Close','Volume']].values.astype(float)
# Normalize features column-wise
means = features.mean(axis=0, keepdims=True)
stds = features.std(axis=0, keepdims=True) + 1e-8
features_norm = (features - means) / stds

X = torch.tensor(features_norm, dtype=torch.float32)
y = torch.tensor(df['Direction'].values.reshape(-1,1), dtype=torch.float32)

print('Examples:', X.shape[0], 'features dim:', X.shape[1])
print('Class balance: up =', int(y.sum().item()), 'down =', int(len(y)-y.sum().item()))


In [ ]:
# Model, training setup
input_dim = X.shape[1]
model = BinaryImprint(input_dim=input_dim, hidden_dim=32, imprint_strength=0.2)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

epochs = 200
losses = []
batch_size = None  # full-batch for tiny data; set integer for minibatch

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    preds = model(X)
    loss = criterion(preds, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

# Plot loss
plt.figure(figsize=(8,4))
plt.plot(losses)
plt.title('Training Loss (Binary BCE)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

# Evaluation
model.eval()
with torch.no_grad():
    probs = model(X)
    preds_bin = (probs > 0.5).float()
    acc = (preds_bin.eq(y)).float().mean().item()
    from sklearn.metrics import precision_score, recall_score, f1_score
    y_true = y.numpy().astype(int).ravel()
    y_pred = preds_bin.numpy().astype(int).ravel()
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

print(f"Accuracy: {acc*100:.2f}% | Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f}")


In [ ]:
# Visualize predictions over time
import matplotlib.dates as mdates
times = df.index

plt.figure(figsize=(12,3))
plt.plot(times, df['Return'], label='Return (minute)')
plt.scatter(times, (y.squeeze().numpy()*0), c=y.squeeze().numpy(), cmap='bwr', marker='|', label='True Direction (1=up,0=down)', alpha=0.8)
plt.scatter(times, (preds_bin.squeeze().numpy()*0.01-0.005), c=preds_bin.squeeze().numpy(), cmap='coolwarm', marker='x', label='Predicted Direction', alpha=0.8)
plt.title('Intraday Returns and Direction Labels (true vs predicted)')
plt.xlabel('Time')
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()


## Interpretation & Next steps

- **If accuracy > ~60%:** The Imprint Field shows sensitivity to intraday microstructure — good feasibility signal.
- **If accuracy ≈ 50%:** No signal captured yet; consider tuning `imprint_strength`, hidden_dim, or adding engineered features (lags, VWAP, orderflow proxies).
- **Next:** extract top up/down minute segments and visualize latent activations, or run the experiment across multiple tickers/dates.

---
**How to open in Colab:**
1. Download the notebook file from the link provided below.
2. In Colab: File → Upload notebook → Select the downloaded file.

Alternatively, you can copy-paste cells into a fresh Colab notebook and run sequentially.